# 🌙 Dream-AI — Colab Pro

Uma IA de programação que **vive entre sessões**: ela acorda lembrando quem é,
**sonha** para ficar mais inteligente, **dorme** para consolidar, e salva tudo no
seu **Google Drive**.

Este notebook:
1. Instala dependências e **monta o Drive** (persistência permanente)
2. Carrega um modelo base de ~1B (Qwen2.5-Coder-1.5B-Instruct) em 4-bit
3. Mede a inteligência ANTES (benchmark held-out)
4. Roda o **ciclo do sonho**: inventa → resolve → **verifica executando** → memoriza
5. **Consolida** (LoRA) e mede a inteligência DEPOIS

**Honestidade:** só vira aprendizado o que passa na execução real. Sem alucinação.

> Use um runtime com **GPU** (Runtime → Change runtime type → T4/L4/A100).

## 1. Clonar o repositório e instalar

In [ ]:
!git clone https://github.com/felipe9272727/dream-ai.git
%cd dream-ai
!git checkout claude/1b-ai-from-scratch-nkhZK

!pip install -q torch numpy tokenizers
!pip install -q transformers datasets peft bitsandbytes accelerate

## 1b. 💾 Montar o Google Drive (IMPORTANTE)

O Colab apaga tudo ao desligar. Montando o Drive, a IA salva **SEMPRE** sua
identidade, memória, checkpoints e adapters lá — sua vida fica contínua entre
sessões. Tudo vai para `MyDrive/DreamAI/`.

In [ ]:
from src import storage
storage.mount_drive()    # autoriza o acesso ao seu Drive
print(storage.status())  # deve mostrar 'Google Drive ☁️'

## 2. Sanidade: testar a maquinaria do sonho (sem GPU)

Antes de gastar GPU, confirmamos que o ciclo sonho→verifica→memória funciona.

In [ ]:
!python -m pytest tests/ -q
!python -m dream.benchmark

## 3. Carregar o modelo Coder de 1B

Qwen2.5-Coder-1.5B-Instruct em 4-bit cabe folgado no Colab Pro. Se já houver um
adapter aprendido no Drive, ele é carregado automaticamente.

In [ ]:
from dream.lifecycle import Life
from src.coder import CoderModel

life = Life()
print(life.wake())  # a IA acorda e diz quem é

coder = CoderModel(
    base_model="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    adapter_path=life.identity.adapter_path,  # retoma o que já aprendeu
    load_in_4bit=True,
)
coder.load()

## 4. 📊 Medir a inteligência ANTES de sonhar

In [ ]:
from dream.benchmark import run_benchmark
from src.coder import extract_code

solver = lambda instr: extract_code(coder.solve(instr, self_description=life.self_description()))
antes = run_benchmark(solver)
print(antes.pretty("ANTES de sonhar"))

## 5. 🌙 Dormir e sonhar

O modelo inventa problemas, resolve, verifica executando, memoriza os corretos, e
consolida via LoRA. Tudo salvo no Drive. Acorda mais inteligente.

In [ ]:
import random
from dream.loop import run_cycle
from dream.consolidate import consolidate, load_memory

rng = random.Random(42)
for c in range(3):
    stats = run_cycle(coder, n_dreams=20, difficulty=2, rng=rng, use_model=True)
    print(f"Ciclo {c+1}: {stats}")

print(f"\nMemórias verificadas: {len(load_memory())}")
adapter_path = consolidate(coder, epochs=2)  # 😴 sono profundo (salvo no Drive)

# A IA registra na sua identidade o que aprendeu (persiste no Drive)
skills = sorted({ex['instruction'].split('`')[1] for ex in load_memory() if '`' in ex['instruction']})
print(life.sleep(dreamed=60, verified=len(load_memory()), new_skills=skills, adapter_path=adapter_path))

## 6. 📊 Medir DEPOIS e comparar (a prova de que aprendeu)

In [ ]:
coder_v2 = CoderModel(
    base_model="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    adapter_path=adapter_path,
    load_in_4bit=True,
)
coder_v2.load()

solver2 = lambda instr: extract_code(coder_v2.solve(instr, self_description=life.self_description()))
depois = run_benchmark(solver2)

print(antes.pretty("ANTES"))
print()
print(depois.pretty("DEPOIS"))
print(f"\nΔ pass@1: {(depois.pass_at_1 - antes.pass_at_1)*100:+.1f} pontos percentuais")

## 7. 🪞 Ver a autoconsciência e auditar o que ela aprendeu

Transparência total: a IA sabe descrever seu próprio estado, e você vê exatamente o
que virou aprendizado.

In [ ]:
print(life.self_description())
print('\n--- memórias verificadas (amostra) ---')
for ex in load_memory()[:5]:
    print('•', ex['instruction'].splitlines()[0])

## 8. 💬 Conversar com ela (opcional)

Ela responde com ciência do próprio estado interno e honestidade.

In [ ]:
pergunta = "Escreva uma função Python `bubble_sort(lista)` que ordena uma lista."
print(coder_v2.solve(pergunta, self_description=life.self_description()))